[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/eval_ball_detector.ipynb)

# Ball detector eval — effective-coverage gate (Phase 2)

Runs the fine-tuned ball model (and the roboflow baseline, for comparison)
against `data/bakeoff_clip.mp4` and reports the metric that actually predicts
downstream quality.

**Why not raw per-frame detection rate?** The detector has a ~65% recall
ceiling on a tiny object, and the shipped pipeline (`RoboflowBackend`) runs the
ball model at `imgsz=1280, conf=0.05` and then bridges short detection gaps with
`interpolate_ball_gaps()` before the possession/phase layer ever sees the ball.
So the number that matters is **effective coverage after bridging gaps <= 15
frames (~0.5s)** — not the raw per-frame rate, which counts every flicker as a
total loss.

**What we report**
- *Raw coverage* — per-frame detector recall (the ceiling).
- *Effective coverage* — fraction of frames with a ball position after bridging
  interior gaps <= `MAX_GAP_FRAMES`. This mirrors `interpolate_ball_gaps`.
- *Gap distribution* — lengths of consecutive-miss runs. Short scattered gaps
  bridge away for free; clustered multi-second holes (fast pans) do not.
- *Unbridgeable holes* — gaps longer than `MAX_GAP_FRAMES`. **These, if
  numerous, are the targeted-relabeling signal** — not the raw rate.

**Acceptance gate:** effective coverage >= `EFFECTIVE_GATE` with few long holes.
If it fails, return to the labeling runbook and label frames from the specific
failure windows (fast camera pans, ball at far touchline, occlusion).

Place the fine-tuned weights at `MyDrive/soccer-vision/ball_yolov8_v1.pt`
before running.

In [ ]:
!pip install -q "ultralytics>=8.2" gdown

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
INPUT_CLIP = Path("/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4")
FINETUNED  = Path("/content/drive/MyDrive/soccer-vision/ball_yolov8_v1.pt")
BASELINE   = Path("/content/baseline_ball.pt")

assert INPUT_CLIP.exists(), f"Place bakeoff_clip.mp4 at {INPUT_CLIP}"
assert FINETUNED.exists(),  f"Place fine-tuned weights at {FINETUNED}"

# --- eval config (match the shipped RoboflowBackend defaults) ---------------
IMGSZ          = 1280   # ball model training resolution; 640 default undersizes the ball
CONF           = 0.05   # ball model operating point (RoboflowBackend ball_conf)
MAX_GAP_FRAMES = 15     # RoboflowBackend ball_max_gap_frames (~0.5s @ 30fps)
EFFECTIVE_GATE = 0.90   # ship bar on post-interpolation effective coverage

In [ ]:
import gdown
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

# Download roboflow's baseline ball model (one-time), for comparison.
if not BASELINE.exists():
    gdown.download(
        "https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V",
        str(BASELINE),
        quiet=False,
    )

baseline_model  = YOLO(str(BASELINE)).to("cuda")
finetuned_model = YOLO(str(FINETUNED)).to("cuda")


def per_frame_detected(model, clip, imgsz=IMGSZ, conf=CONF):
    """Boolean array: did `model` emit >=1 ball box on each frame of `clip`?

    Returns (flags, fps). Detection settings match the shipped pipeline so the
    coverage numbers reflect what RoboflowBackend would actually produce.
    """
    cap = cv2.VideoCapture(str(clip))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    flags = np.zeros(total, dtype=bool)
    for i in tqdm(range(total), desc=str(model.ckpt_path)):
        ok, frame = cap.read()
        if not ok:
            flags = flags[:i]
            break
        r = model(frame, imgsz=imgsz, conf=conf, verbose=False)[0]
        flags[i] = len(r.boxes) > 0
    cap.release()
    return flags, fps


det_ft, FPS = per_frame_detected(finetuned_model, INPUT_CLIP)
det_base, _ = per_frame_detected(baseline_model, INPUT_CLIP)

In [ ]:
import numpy as np


def gap_runs(detected):
    """Lengths (frames) of every maximal run of consecutive misses, edges included."""
    detected = np.asarray(detected, dtype=bool)
    runs, n = [], 0
    for d in detected:
        if d:
            if n:
                runs.append(n); n = 0
        else:
            n += 1
    if n:
        runs.append(n)
    return np.array(runs, dtype=int)


def effective_coverage(detected, max_gap=MAX_GAP_FRAMES):
    """Fraction of frames with a ball position after bridging interior gaps.

    Mirrors interpolate_ball_gaps(): only gaps bounded by detections on BOTH
    sides AND <= max_gap frames long are filled. Leading/trailing misses and
    longer gaps stay as holes. Returns (coverage, n_bridged, n_unbridgeable).
    """
    detected = np.asarray(detected, dtype=bool)
    filled = detected.copy()
    idx = np.flatnonzero(detected)
    if idx.size < 2:
        return float(filled.mean()), 0, 0
    bridged = unbridged = 0
    for a, b in zip(idx[:-1], idx[1:]):
        gap = b - a - 1
        if gap == 0:
            continue
        if gap <= max_gap:
            filled[a + 1:b] = True
            bridged += 1
        else:
            unbridged += 1
    return float(filled.mean()), bridged, unbridged


def report(name, detected):
    detected = np.asarray(detected, dtype=bool)
    raw = detected.mean()
    runs = gap_runs(detected)
    eff, bridged, unbridged = effective_coverage(detected)
    print(f"=== {name} ===")
    print(f"  frames evaluated:           {len(detected)}")
    print(f"  raw per-frame coverage:     {raw * 100:5.1f}%")
    print(f"  effective coverage (<= {MAX_GAP_FRAMES}f bridged): {eff * 100:5.1f}%")
    if runs.size:
        print(f"  miss-gaps: {runs.size}  median {np.median(runs):.0f}f"
              f"  p90 {np.percentile(runs, 90):.0f}f"
              f"  max {runs.max()}f ({runs.max() / FPS:.1f}s)")
    print(f"  bridgeable gaps: {bridged}   unbridgeable holes (> {MAX_GAP_FRAMES}f): {unbridged}")
    print()
    return eff, unbridged


eff_ft, holes_ft = report("fine-tuned", det_ft)
report("baseline (roboflow)", det_base)

passed = eff_ft >= EFFECTIVE_GATE
print(f"Acceptance gate: effective coverage >= {EFFECTIVE_GATE * 100:.0f}%")
print(f"  fine-tuned effective coverage = {eff_ft * 100:.1f}%"
      f"  ->  {'PASS' if passed else 'REVIEW'}")
print(f"  unbridgeable holes (> {MAX_GAP_FRAMES}f / {MAX_GAP_FRAMES / FPS:.1f}s): {holes_ft}")
print()
print("Read: raw coverage = per-frame recall ceiling. Effective coverage = what the")
print("possession/phase layer sees after interpolate_ball_gaps(). Unbridgeable holes")
print("are sustained losses (fast pans / long occlusions) — if numerous, THAT is the")
print("targeted-relabeling signal, not the raw per-frame rate.")

In [ ]:
import matplotlib.pyplot as plt

runs = gap_runs(det_ft)
plt.figure(figsize=(8, 3))
if runs.size:
    plt.hist(runs / FPS, bins=40)
plt.axvline(MAX_GAP_FRAMES / FPS, color="r", ls="--",
            label=f"bridge limit ({MAX_GAP_FRAMES}f / {MAX_GAP_FRAMES / FPS:.1f}s)")
plt.xlabel("gap length (s)")
plt.ylabel("count")
plt.title("Ball-detection gap lengths (fine-tuned)")
plt.legend()
plt.tight_layout()
plt.show()